# QAOA for Max-Cut

QAOA's reference problem, and the best place to start: Max-Cut is **unconstrained**
and its objective is **positive**, so nothing about penalties, feasibility or
signed ratios gets in the way of seeing what the algorithm does.

Companion to [`tutorials/02-optimization/01-qaoa-maxcut.md`](../tutorials/02-optimization/01-qaoa-maxcut.md).

**Requires:** `pip install -e ".[qiskit]"`

## 1. The problem

Split the vertices into two sets so as many edges as possible run *between* them.
Edge `(i,j)` is cut when `x_i != x_j`, which `x_i + x_j - 2 x_i x_j` scores as
exactly 1 or 0.

In [1]:
from qprac_lab.algorithms.optimization.qaoa_maxcut import make_maxcut_graph
from qprac_lab.algorithms.optimization.qubo_builder import maxcut_qubo

edges = make_maxcut_graph(num_nodes=8, degree=3, seed=42)
qubo = maxcut_qubo(8, edges)

print(f"8 nodes, {len(edges)} edges, 3-regular")
print(f"edges: {edges}")
print(f"maximum cut (brute force): {-qubo.brute_force()['objective_value']:.0f}")

8 nodes, 12 edges, 3-regular
edges: [(0, 1), (0, 2), (0, 5), (1, 4), (1, 7), (2, 3), (2, 5), (3, 7), (3, 6), (4, 6), (4, 5), (6, 7)]
maximum cut (brute force): 10


Under `x = (1 - z)/2` each edge becomes `(1 - Z_i Z_j)/2` — the Ising form every
QAOA paper writes down.

In [2]:
cost_operator, offset = qubo.to_ising()
print(f"qubits: {cost_operator.num_qubits}, terms: {len(cost_operator)}, offset: {offset:.1f}")

qubits: 8, terms: 12, offset: -6.0


## 2. Run QAOA

In [3]:
from qprac_lab.algorithms.optimization.qaoa_maxcut import run_qaoa_maxcut_tutorial

result = run_qaoa_maxcut_tutorial(reps=3)

print(f"objective evaluations: {result.function_evaluations}")
print(f"maximum cut (exact):   {result.max_cut_value}")
print(f"QAOA best sample:      {result.best_cut_value}  (ratio {result.approximation_ratio:.4f})")
print(f"QAOA expected cut:     {result.expected_cut_value:.4f}  "
      f"(ratio {result.expected_approximation_ratio:.4f})   <- the headline")

objective evaluations: 300
maximum cut (exact):   10
QAOA best sample:      10  (ratio 1.0000)
QAOA expected cut:     9.0498  (ratio 0.9050)   <- the headline


**Read the expected value, not the best sample.** QAOA returns a distribution,
and its guarantees are statements about `<C>`. "Best of 4096 shots" measures your
shot budget as much as your algorithm — it improves indefinitely just by sampling
more, which is exactly why it reads 1.000 here.

## 3. Baselines

The random-assignment row is not arbitrary: a uniformly random cut separates each
edge with probability 1/2, so it scores `|E|/2` in expectation. **Anything that
cannot beat that has done nothing.**

In [4]:
for name, payload in result.baseline_report.items():
    value = payload.get("cut_value", payload.get("expected_cut_value"))
    print(f"{name:>20}: cut {value:>5}  ratio {payload['ratio']:.4f}")
print(f"\nP(sampling an optimal cut): {result.optimal_probability:.1%}")
print(f"beats random guessing:      {result.beats_random_guessing}")

         brute_force: cut    10  ratio 1.0000
              greedy: cut    10  ratio 1.0000
   random_assignment: cut   6.0  ratio 0.6000

P(sampling an optimal cut): 49.7%
beats random guessing:      True


0.905 expected ratio against a 0.600 random baseline, with half of all shots on
an optimal cut — a genuinely good QAOA result.

And then the deflating line: **greedy also found a cut of 10**, instantly. On a
graph this size the quantum method has nothing to offer. That is not a flaw in the
implementation; it is what an 8-vertex problem looks like.

## 4. Does depth help?

In [5]:
print(f"{'p':>3} {'E[ratio]':>10} {'P(optimal)':>12}")
for reps in (1, 2, 3, 5):
    run = run_qaoa_maxcut_tutorial(reps=reps, shots=2048)
    print(f"{reps:>3} {run.expected_approximation_ratio:>10.4f} {run.optimal_probability:>11.1%}")

  p   E[ratio]   P(optimal)
  1     0.7978       14.1%


  2     0.8730       33.3%


  3     0.9030       49.5%


  5     0.9532       73.8%


## 5. Max-Cut is the most noise-robust method here

This is the flip side of the VQE notebook. VQE loses seven orders of magnitude of
accuracy to light noise; Max-Cut barely moves.

The reason is structural: **Max-Cut needs only the *ranking* of bitstrings to
survive, not precise amplitudes.** Noise flattens the distribution without
necessarily reordering its peak. VQE needs an expectation value good to `1e-3`,
and depolarizing noise attacks exactly that.

(Noisy 8-qubit simulation is slow — Aer propagates a density matrix — so this uses
a reduced optimiser budget.)

In [6]:
print(f"{'noise':>10} {'E[ratio]':>10} {'P(optimal)':>12}")
for noise in [None, "light", "moderate"]:
    run = run_qaoa_maxcut_tutorial(
        reps=2, backend="aer", noise=noise, shots=2048, maxiter=60,
    )
    print(f"{noise or 'ideal':>10} {run.expected_approximation_ratio:>10.4f} "
          f"{run.optimal_probability:>11.1%}")
print(f"\nrandom-guessing baseline: {result.random_guess_ratio:.4f}")

     noise   E[ratio]   P(optimal)


     ideal     0.8799       34.6%


     light     0.8729       32.8%


  moderate     0.8393       26.6%

random-guessing baseline: 0.6000


> **The pattern worth taking away:** algorithms that need a precise *number* break
> first; algorithms that need only an *ordering* last longest.

Full sweep across all four tutorials:
[`tutorials/05-benchmarking/noise_benchmark.md`](../tutorials/05-benchmarking/noise_benchmark.md).

## When not to use this

- **On small graphs.** Greedy tied the exact optimum in microseconds.
- **Where good heuristics exist**, which is most graphs. The real bar is not brute
  force — it is Goemans–Williamson, which *guarantees* 0.878 in the worst case.
  Reaching 0.905 on one easy 8-vertex graph is not evidence of beating it.
- **At shallow depth on hard instances**, and depth is what noisy hardware cannot
  afford.